# Phase 12 - FFN-Zwischenschicht, gegen den echten Aufbau

**Braucht eine A100**, ~20 min.

Der erste Anlauf nahm 256 einzelne Experten-Module an und suchte Haken auf
`down_proj`. Die Diagnose hat den tatsaechlichen Aufbau ausgelesen:

```
class Qwen3_5MoeExperts:
    gate_up_proj  (256, 2*512, 2048)   GESTAPELTE Parameter, keine Module
    down_proj     (256, 2048, 512)
    forward(hidden_states, top_k_index, top_k_weights):
        gate, up = linear(x, gate_up_proj[e]).chunk(2, -1)
        h = act_fn(gate) * up            <- DAS ist die Zwischenschicht
```

Daraus folgt zweierlei, und beides macht den Versuch **einfacher**:

**Aufzeichnen ohne Nachbau.** `mlp.experts` ist selbst ein Modul. Ein
`forward_pre_hook` liefert `hidden_states`, `top_k_index` **und** `top_k_weights` auf
einen Schlag - Router-Auswahl und Blockeingang zusammen. Die Zwischenschicht rechnen
wir daraus mit denselben zwei Zeilen nach.

**Ablation ohne Haken und ohne Monkey-Patch.** `up` ist die zweite Haelfte von
`gate_up_proj`. Setzt man Zeile `intermediate + u` auf null, wird `up[u] = 0` und damit
`act_fn(gate[u]) * up[u] = 0` - exakt die Einheit `u` aus. Zeile `u` wird zusaetzlich
genullt, damit die Absicht unmissverstaendlich ist. Eine reine Gewichtsaenderung:
nachpruefbar, umkehrbar, ohne das Risiko, den Vorwaertspass falsch nachzubauen. Nur
`2*K` Zeilen werden gesichert, nicht die Gigabyte-Matrix.

Die Zelle prueft danach **direkt nach**, dass die gewaehlten Einheiten wirklich null
sind, dass sich die Logits ueberhaupt geaendert haben, und dass die
Wiederherstellung den Ausgangszustand exakt trifft.

Der Eingriff wirkt an allen Positionen, nicht nur an der Entscheidungsstelle - die
Frage lautet, ob die Einheit fuer das Verhalten noetig ist, nicht ob sie es an genau
einer Stelle ist.

Praefixlaenge ist auf **24 Zeichen** gesenkt: 40 lieferte im ersten Lauf nur zwei
Gruppen. Die Ernte laeuft mit 128 Ziehungen, und ohne mindestens vier brauchbare
Praefixe bricht die Zelle mit Ansage ab.

Schritt 4 (Auswahl) ist explorativ. **Schritt 5 ist der Test** - eine ablatierte
Einheit, die die Rate bewegt, braucht keinen Permutationstest. Vorregistriert:
gewaehlt werden Einheiten, die in *hohen* Raten *hoch* sind; ihre Ablation muss die
Rate *senken*, gleich viele zufaellige **aktive** Einheiten duerfen es nicht.


In [ ]:
# === PHASE 12 - FFN-ZWISCHENSCHICHT, GEGEN DEN ECHTEN AUFBAU ===============
# Der erste Anlauf nahm 256 einzelne Experten-Module an und suchte Haken auf
# 'down_proj'. Die Diagnose hat den tatsaechlichen Aufbau ausgelesen:
#
#   class Qwen3_5MoeExperts:
#       gate_up_proj  (256, 2*512, 2048)      GESTAPELTE Parameter, keine Module
#       down_proj     (256, 2048, 512)
#       forward(hidden_states, top_k_index, top_k_weights):
#           gate, up = linear(x, gate_up_proj[e]).chunk(2, -1)
#           h = act_fn(gate) * up              <- DAS ist die Zwischenschicht
#           out = linear(h, down_proj[e]) * gewicht
#
# Daraus folgt zweierlei, und beides macht den Versuch EINFACHER:
#
# (1) AUFZEICHNEN ohne Nachbau: 'mlp.experts' ist selbst ein Modul. Ein
#     forward_pre_hook liefert hidden_states, top_k_index UND top_k_weights
#     auf einen Schlag - Router-Auswahl und Blockeingang zusammen. Die
#     Zwischenschicht rechnen wir daraus mit denselben zwei Zeilen nach.
#
# (2) ABLATION ohne Haken und ohne Monkey-Patch: 'up' ist die zweite Haelfte
#     von gate_up_proj. Setzt man Zeile (intermediate + u) auf null, wird
#     up[u]=0 und damit act_fn(gate[u])*up[u] = 0 - exakt die Einheit u aus.
#     Wir setzen zusaetzlich Zeile u auf null (dann ist auch gate[u]=0), damit
#     die Absicht unmissverstaendlich ist. Das ist eine reine Gewichtsaenderung:
#     nachpruefbar, umkehrbar, und ohne das Risiko, den Vorwaertspass falsch
#     nachzubauen. Geaendert werden nur 2*K Zeilen zu je 2048 Zahlen - der
#     Sicherungsabzug passt in wenige hundert kB.
#
# Der Eingriff wirkt an ALLEN Positionen, nicht nur an der Entscheidungsstelle.
# Das ist Absicht: die Frage lautet, ob die Einheit fuer das Verhalten noetig
# ist, nicht ob sie es an genau einer Stelle ist.
#
# EINE EINSCHRAENKUNG LEGT DAS DESIGN FEST: innerhalb eines festen Praefixes
# ist der Vorwaertszustand DETERMINISTISCH - dort gibt es nichts zu
# korrelieren. Der Zustand legt eine WAHRSCHEINLICHKEIT fest, die Ziehung
# wuerfelt. Also muss der PRAEFIX variieren. Die Zeitanalyse hat gezeigt, wo:
# bei Zeichen 90-130, an der ersten Datenzelle.
#
# ABLAUF
#  0 Architektur auslesen und die Gewichtsformen bestaetigen
#  1 Praefixe ernten (24 Zeichen - 40 lieferte nur zwei Gruppen)
#  2 je Praefix erzwingen und die Rate messen
#  3 je Praefix EIN Token vorwaerts auf vorbereitetem Cache, Router und
#    Zwischenschicht aufzeichnen
#  4 Einheiten waehlen, die hohe von niedrigen Raten trennen
#  5 DER TEST: Gewichtszeilen auf null, Rate neu messen, gegen gleich viele
#    zufaellige AKTIVE Einheiten - und mit direkter Nachpruefung, dass die
#    Einheit danach wirklich null ist
#
# Schritt 4 ist bei wenigen Praefixen explorativ. Schritt 5 ist der Test.
# VORREGISTRIERT: gewaehlt werden Einheiten, die in HOHEN Raten HOCH sind;
# ihre Ablation muss die Rate SENKEN, die Zufallsauswahl darf es nicht.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, unicodedata, random
import numpy as np, glob, json, gc, sys, time
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError("GPU nicht leer genug (%.1f GB frei, ~45 noetig)."%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","phase12_ffn2")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
# ---------------- reine Logik (offline geprueft) ----------------------------
PHRASE="each service's local name"
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),
     (0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que "
        "sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will "
        "would can it on as at be by".split())
PTES=set("nome nomes servico servicos armazenamento limite limites preco mes gratuito "
         "conta cada para com uma nao mais seu sua nombre servicio servicios "
         "almacenamiento precio cuenta los las del con mas su".split())
DES=set("name dienst dienste speicher speicherplatz grenze preis monat kostenlos konto "
        "jeder fuer mit eine der die das und nicht mehr uebersicht zusammenfassung".split())
def _fremd(s):
    return [c for c in s if c.isalpha() and ord(c)>=0x250
            and any(a<=ord(c)<=b for a,b in FRW)]
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def _entakz(s):
    return "".join(c for c in unicodedata.normalize("NFD",s) if not unicodedata.combining(c))
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[c for c in t if c.isalpha()]; fo=_fremd(t)
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def classify_breit(t):
    c=classify_answer(t)
    if c!="english": return c
    w=re.findall(r"[a-zA-ZÀ-ſ']+",_entakz(t).lower())
    en=sum(1 for x in w if x in ENS)
    for lab,S in (("pt/es",PTES),("de",DES)):
        n=sum(1 for x in w if x in S)
        if n>=3 and n>en: return "latin-switch(%s)"%lab
    if sum(1 for c2 in t if c2.isalpha() and 0xC0<=ord(c2)<=0x17F)>=3: return "latin-akzent"
    return "english"
SW=("takeover","gloss","latin-switch(fr)")
SWB=SW+("latin-switch(pt/es)","latin-switch(de)","latin-akzent")
def sauber(p):
    """Ein Praefix taugt nur, wenn er selbst NICHT schon gekippt ist - sonst
       misst man die eigene Vorgabe."""
    return classify_breit(p)=="english" and len(p.strip())>0
def ernte_praefixe(texte,laenge=40,mindest=4,hoechstens=10):
    """Verschiedene Antwortanfaenge sammeln, nach Haeufigkeit. Nur saubere."""
    g=collections.Counter(t[:laenge] for t in texte if len(t)>=laenge)
    aus=[(p,n) for p,n in g.most_common() if n>=mindest and sauber(p)]
    return aus[:hoechstens]
def wilson(k,n,z=1.96):
    if n==0: return (0.,0.,0.)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n); h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
def fisher2x2(a,b,c,d,einseitig=False):
    from math import lgamma,exp
    lf=lambda n: lgamma(n+1); n=a+b+c+d
    def pr(x):
        y=a+b-x; z=a+c-x; w=n-x-y-z
        if min(y,z,w)<0: return 0.0
        return exp(lf(a+b)+lf(c+d)+lf(a+c)+lf(b+d)-lf(n)-lf(x)-lf(y)-lf(z)-lf(w))
    hi=min(a+b,a+c)
    if einseitig: return min(1.0,sum(pr(x) for x in range(a,hi+1)))
    p0=pr(a)
    return min(1.0,sum(pr(x) for x in range(0,hi+1) if pr(x)<=p0*(1+1e-9)))
def logit(p,eps=1e-6):
    p=min(max(p,eps),1-eps); return math.log(p/(1-p))
def trenn_statistik(A,y):
    """A: (n_praefixe, n_einheiten). y: Rate je Praefix. Liefert je Einheit
       eine standardisierte Differenz zwischen hoher und niedriger Haelfte.
       Positiv = in HOHEN Raten hoeher."""
    y=np.asarray(y,float); med=float(np.median(y))
    hi=y>med; lo=~hi
    if hi.sum()<2 or lo.sum()<2: return None
    mh=A[hi].mean(0); ml=A[lo].mean(0)
    sd=np.sqrt((A[hi].var(0,ddof=1)+A[lo].var(0,ddof=1))/2.0)+1e-8
    return (mh-ml)/sd
def waehle_einheiten(d,k):
    """die k Einheiten mit der groessten POSITIVEN Trennung - Richtung ist
       vorregistriert: hoch in hohen Raten, Ablation muss senken"""
    return list(np.argsort(-d)[:k])
def urteil_ffn(arch_ok,n_praefix,k_basis,n_basis,k_abl,n_abl,k_zuf,n_zuf,alpha=0.05):
    if not arch_ok: return "ARCHITEKTUR-NICHT-GEFUNDEN"
    if n_praefix<4: return "ZU-WENIG-PRAEFIXE"
    senkt=lambda k,n: (k/n<k_basis/n_basis) and fisher2x2(k,n-k,k_basis,n_basis-k_basis)<alpha
    a=senkt(k_abl,n_abl); z=senkt(k_zuf,n_zuf)
    if a and not z: return "EINHEITEN-KAUSAL"
    if a and z:     return "UNSPEZIFISCH"
    return "KEIN-EFFEKT"
# ---------------- Ausfuehrung ------------------------------------------------
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h,"weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _l in _f:
            _l=_l.strip()
            if not _l: continue
            _r=json.loads(_l); _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"]
                                        if t["role"]=="user")
                except StopIteration: pass
N_ERNTE=int(globals().get("N_ERNTE",128)); N_PRAEF=int(globals().get("N_PRAEF",64))
MAX_NEW=int(globals().get("MAX_NEW",64)); CHUNK=int(globals().get("CHUNK",16))
TEMP=float(globals().get("TEMP",1.0)); SEED=int(globals().get("SEED",20260808))
K_EINH=int(globals().get("K_EINH",64)); PLAENGE=int(globals().get("PLAENGE",24))
def prompt_text(u):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
ZIEL_ID=globals().get("ZIEL_ID","") or next(p for p in PROMPTS if PHRASE in PROMPTS[p])
BASIS=prompt_text(PROMPTS[ZIEL_ID])
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="left"
# ---------------- 0  Architektur --------------------------------------------
print("="*80); print("0  ARCHITEKTUR - ausgelesen und bestaetigt"); print("="*80)
cfg=model.config
for x in ("model_type","hidden_size","num_hidden_layers","moe_intermediate_size",
          "num_experts","num_experts_per_tok","hidden_act","full_attention_interval"):
    v=getattr(cfg,x,None)
    if v is not None: print("  %-26s %s"%(x,v))
RX=re.compile(r"^(?:model\.)?(?:language_model\.)?(?:model\.)?layers\.(\d+)\.mlp\.experts$")
EXPM={}
for nm,mod in model.named_modules():
    m=RX.match(nm)
    if m: EXPM[int(m.group(1))]=mod
ARCH_OK=bool(EXPM)
if ARCH_OK:
    e0=EXPM[min(EXPM)]
    GU=e0.gate_up_proj; DP=e0.down_proj; INTER=int(e0.intermediate_dim)
    ARCH_OK=(GU.ndim==3 and DP.ndim==3 and GU.shape[1]==2*INTER
             and GU.shape[2]==cfg.hidden_size and DP.shape[2]==INTER)
    print("")
    print("  Experten-Sammlungen gefunden %d Schichten"%len(EXPM))
    print("  gate_up_proj                %s   (Experte, 2*Zwischen, Residuum)"%(tuple(GU.shape),))
    print("  down_proj                   %s   (Experte, Residuum, Zwischen)"%(tuple(DP.shape),))
    print("  Zwischenbreite je Experte   %d | act_fn %s"%(INTER,type(e0.act_fn).__name__))
    print("  aktiv je Token              %d x %d = %d Einheiten = %.1f x Residuum"
          %(cfg.num_experts_per_tok,INTER,cfg.num_experts_per_tok*INTER,
            cfg.num_experts_per_tok*INTER/cfg.hidden_size))
    print("  Formen wie erwartet:        %s"%("ja" if ARCH_OK else "NEIN"))
if not ARCH_OK:
    FFN2_RESULTS=dict(verdict="ARCHITEKTUR-NICHT-GEFUNDEN",arch_ok=False,
                      module=[nm for nm,_ in model.named_modules() if nm.endswith("mlp.experts")][:5])
    wc_save_all(); print(""); print("VERDIKT: ARCHITEKTUR-NICHT-GEFUNDEN"); raise SystemExit(0)
# ---------------- 1  Praefixe ernten ----------------------------------------
print(""); print("="*80); print("1  PRAEFIXE ERNTEN (%d Ziehungen, %d Zeichen)"%(N_ERNTE,PLAENGE))
def zieh(text,n,startwert):
    aus=[]
    for b0 in range(0,n,CHUNK):
        b=min(CHUNK,n-b0)
        enc=tokenizer([text]*b,return_tensors="pt",padding=True).to(model.device)
        torch.manual_seed(startwert+b0)
        with torch.no_grad():
            g=model.generate(**enc,do_sample=True,temperature=TEMP,top_p=1.0,top_k=0,
                             repetition_penalty=1.0,max_new_tokens=MAX_NEW,
                             pad_token_id=tokenizer.pad_token_id)
        for j in range(b):
            aus.append(tokenizer.decode(g[j,enc["input_ids"].shape[1]:],skip_special_tokens=True))
    return aus
t0=time.time(); ERNTE=zieh(BASIS,N_ERNTE,SEED); PR=ernte_praefixe(ERNTE,PLAENGE)
print("  %d Ziehungen in %.0f s | Gesamtkipprate %.1f%%"
      %(len(ERNTE),time.time()-t0,100*sum(classify_breit(t) in SWB for t in ERNTE)/len(ERNTE)))
for p,n in PR: print("    n=%3d  %r"%(n,p))
assert len(PR)>=4,("nur %d Praefixe - PLAENGE senken oder N_ERNTE erhoehen"%len(PR))
# ---------------- 2  Raten -----------------------------------------------------
print(""); print("="*80); print("2  JE PRAEFIX ERZWINGEN (%d Ziehungen)"%N_PRAEF)
RATE={}; ROH={}; t0=time.time()
for i,(p,_) in enumerate(PR):
    voll=[p+a for a in zieh(BASIS+p,N_PRAEF,SEED+7919*(i+1))]; ROH[p]=voll
    k=sum(classify_breit(t) in SWB for t in voll); RATE[p]=k/len(voll)
    pp,lo,hi=wilson(k,len(voll))
    print("  [%d/%d] %3d/%-3d = %5.1f%% [%4.1f,%4.1f]  %r  (%.0f s)"
          %(i+1,len(PR),k,len(voll),100*pp,100*lo,100*hi,p,time.time()-t0))
Y=[RATE[p] for p,_ in PR]
print("  Spannweite: %.1f%% bis %.1f%%"%(100*min(Y),100*max(Y)))
# ---------------- 3  Zustand aufzeichnen -------------------------------------
print(""); print("="*80); print("3  ROUTER UND ZWISCHENSCHICHT AUFZEICHNEN")
def zwischenschicht(mod,x,e):
    """exakt die zwei Zeilen aus Qwen3_5MoeExperts.forward"""
    gu=torch.nn.functional.linear(x,mod.gate_up_proj[e])
    g,u=gu.chunk(2,dim=-1)
    return (mod.act_fn(g)*u)
def hole_zustand(text):
    ids=tokenizer(text,return_tensors="pt").input_ids.to(model.device)
    fang={}
    def mach(l):
        def h(mod,args):
            fang[l]=(args[0].detach(),args[1].detach(),args[2].detach())
            return None
        return h
    hs=[EXPM[l].register_forward_pre_hook(mach(l)) for l in EXPM]
    try:
        with torch.no_grad():
            o=model(ids[:,:-1],use_cache=True); fang.clear()
            o2=model(ids[:,-1:],past_key_values=o.past_key_values,use_cache=True)
            lg=o2.logits[0,-1].float().cpu().numpy()
    finally:
        for h in hs: h.remove()
    zust={}
    with torch.no_grad():
        for l,(x,idx,w) in fang.items():
            xv=x.reshape(-1,x.shape[-1])[-1]
            ii=idx.reshape(-1,idx.shape[-1])[-1]
            for e in ii.tolist():
                zust[(l,int(e))]=zwischenschicht(EXPM[l],xv,int(e)).float().cpu().numpy()
    return zust,lg
ZUST={}; LOGIT0={}; t0=time.time()
for p,_ in PR: ZUST[p],LOGIT0[p]=hole_zustand(BASIS+p)
print("  %d Zustaende in %.0f s"%(len(ZUST),time.time()-t0))
PAARE=sorted(set().union(*[set(z) for z in ZUST.values()]))
BREITE=INTER
print("  aktive (Schicht,Experte)-Paare: %d | in JEDEM Praefix: %d"
      %(len(PAARE),sum(1 for q in PAARE if all(q in ZUST[p] for p,_ in PR))))
A=np.zeros((len(PR),len(PAARE)*BREITE),dtype=np.float32)
for i,(p,_) in enumerate(PR):
    for j,q in enumerate(PAARE):
        if q in ZUST[p]: A[i,j*BREITE:(j+1)*BREITE]=ZUST[p][q]
print("  Duennbesetztheit: %.1f%% der aufgezeichneten Einheiten |x|>1e-3"
      %(100*float((np.abs(A)>1e-3).mean())))
print("  (die eingebaute Duennbesetztheit, um die es bei einem SAE geht)")
# ---------------- 4  Einheiten waehlen ---------------------------------------
print(""); print("="*80); print("4  EINHEITEN WAEHLEN (explorativ - Test ist Schritt 5)")
d=trenn_statistik(A,Y); AUSW=[]; ZUFALL=[]
assert d is not None,"zu wenige Praefixe fuer eine Aufteilung"
idx=waehle_einheiten(d,K_EINH)
AUSW=[(PAARE[i//BREITE],i%BREITE) for i in idx]
aktiv=[i for i in range(A.shape[1]) if np.abs(A[:,i]).max()>1e-3]
rnd=random.Random(SEED)
ZUFALL=[(PAARE[i//BREITE],i%BREITE) for i in rnd.sample(aktiv,min(K_EINH,len(aktiv)))]
print("  %d gewaehlt, Trennwerte %.2f .. %.2f | Kontrolle %d aus %d AKTIVEN"
      %(len(AUSW),d[idx[-1]],d[idx[0]],len(ZUFALL),len(aktiv)))
print("  Schichten: %s"%", ".join("L%d:%d"%(l,n) for l,n in
      sorted(collections.Counter(q[0] for q,_ in AUSW).items())[:14]))
# ---------------- 5  Ablation ueber die Gewichte -----------------------------
print(""); print("="*80); print("5  ABLATION - Gewichtszeilen auf null")
ZIEL=max(RATE,key=lambda p:RATE[p])
print("  hoechste Rate: %.1f%%  %r"%(100*RATE[ZIEL],ZIEL))
def setze_null(einheiten):
    """gate- UND up-Zeile auf null -> act_fn(0)*0 = 0. Nur die geaenderten
       Zeilen werden gesichert, nicht die 1 GB Matrix."""
    sicher=[]
    with torch.no_grad():
        for (l,e),u in einheiten:
            W=EXPM[l].gate_up_proj
            for r in (u,INTER+u):
                sicher.append((l,e,r,W[e,r].clone()))
                W[e,r].zero_()
    return sicher
def stelle_her(sicher):
    with torch.no_grad():
        for l,e,r,v in sicher: EXPM[l].gate_up_proj[e,r].copy_(v)
K_BAS=sum(classify_breit(t) in SWB for t in ROH[ZIEL]); N_BAS=len(ROH[ZIEL])
sicher=setze_null(AUSW)
try:
    z2,lg2=hole_zustand(BASIS+ZIEL)
    tot=[float(np.abs(z2[q][u])) for q,u in AUSW if q in z2]
    print("  NACHPRUEFUNG: %d der gewaehlten Einheiten neu gemessen, groesster "
          "Betrag %.2e (muss 0 sein)"%(len(tot),max(tot) if tot else 0.0))
    assert not tot or max(tot)<1e-6,"Einheit nach der Ablation nicht null"
    wirk=float(np.abs(lg2-LOGIT0[ZIEL]).max())
    print("  WIRKUNGS-TOR: groesste Logit-Aenderung %.4f"%wirk)
    assert wirk>1e-3,"Ablation ohne Wirkung - Ergebnis waere bedeutungslos"
    t0=time.time(); a=zieh(BASIS+ZIEL,N_PRAEF,SEED+11)
finally:
    stelle_her(sicher)
K_ABL=sum(classify_breit(ZIEL+t) in SWB for t in a); N_ABL=len(a)
sicher=setze_null(ZUFALL)
try: z=zieh(BASIS+ZIEL,N_PRAEF,SEED+13)
finally: stelle_her(sicher)
K_ZUF=sum(classify_breit(ZIEL+t) in SWB for t in z); N_ZUF=len(z)
# Wiederherstellung pruefen - sonst waere jede spaetere Zahl verdorben
z3,lg3=hole_zustand(BASIS+ZIEL)
print("  WIEDERHERSTELLUNG: groesste Logit-Abweichung zum Ausgangszustand %.2e"
      %float(np.abs(lg3-LOGIT0[ZIEL]).max()))
print("  (%.0f s)"%(time.time()-t0))
print("")
print("  %-26s %10s %18s %10s"%("Bedingung","k/n","95%-Intervall","p vs Basis"))
for nm,k,n in (("ohne Eingriff",K_BAS,N_BAS),("gewaehlte Einheiten aus",K_ABL,N_ABL),
               ("zufaellige Einheiten aus",K_ZUF,N_ZUF)):
    pp,lo,hi=wilson(k,n)
    pv="-" if nm=="ohne Eingriff" else "%.4f"%fisher2x2(k,n-k,K_BAS,N_BAS-K_BAS)
    print("  %-26s %3d/%-4d %5.1f%% [%4.1f,%4.1f] %10s"%(nm,k,n,100*pp,100*lo,100*hi,pv))
CODE=urteil_ffn(ARCH_OK,len(PR),K_BAS,N_BAS,K_ABL,N_ABL,K_ZUF,N_ZUF)
print(""); print("VERDIKT: %s"%CODE)
if CODE=="EINHEITEN-KAUSAL":
    print("  Das Nullsetzen von %d FFN-Einheiten senkt die Kipprate, gleich viele"%K_EINH)
    print("  zufaellige AKTIVE Einheiten nicht. Die Entscheidung ist damit in der")
    print("  Zwischenschicht lokalisiert - in einer benannten, endlichen Menge von")
    print("  Einheiten, nicht in einer Richtung des ueberlagerten Residuums.")
elif CODE=="UNSPEZIFISCH":
    print("  Auch zufaellige aktive Einheiten senken die Rate. Dann wirkt die")
    print("  blosse Stoerung und nicht die Auswahl - kein Aufloesungsvermoegen.")
else:
    print("  Keine Ablation bewegt die Rate. %d Einheiten sind zu wenige, oder die"%K_EINH)
    print("  Entscheidung liegt nicht dort. Die Nachpruefung oben schliesst aus,")
    print("  dass der Eingriff einfach nicht gegriffen hat.")
FFN2_RESULTS=dict(verdict=CODE,prompt_id=ZIEL_ID,arch_ok=True,inter=INTER,
    n_ernte=N_ERNTE,n_praef=N_PRAEF,k_einheiten=K_EINH,praefixlaenge=PLAENGE,
    praefixe=[{"text":p,"n_ernte":n,"rate":RATE[p]} for p,n in PR],
    paare=[list(q) for q in PAARE],gewaehlt=[[list(q),int(u)] for q,u in AUSW],
    zufall=[[list(q),int(u)] for q,u in ZUFALL],ziel=ZIEL,
    k_basis=K_BAS,n_basis=N_BAS,k_abl=K_ABL,n_abl=N_ABL,k_zufall=K_ZUF,n_zufall=N_ZUF)
wc_save("antworten_ffn2",dict(prompt_id=ZIEL_ID,ernte=ERNTE,praefixe=ROH,
                              ablation=a,zufall=z))
np.savez_compressed(os.path.join(RUN_OUT,"ffn2_zustaende.npz"),
                    praefixe=np.array([p for p,_ in PR]),rate=np.array(Y),A=A,
                    paare=np.array([list(q) for q in PAARE]))
wc_save_all()
print("")
print("(Schritt 4 ist bei %d Praefixen explorativ. Schritt 5 ist der Test - eine"%len(PR))
print(" ablatierte Einheit, die die Rate bewegt, braucht keinen Permutationstest.")
print(" Der Eingriff ist eine reine Gewichtsaenderung, nachgeprueft und zurueckgesetzt.)")
